# 글자+단어 BiLSTM NER — 토큰 단위 시퀀스 라벨링 (실제 캐글 제출)

**연습 기법** (IOAI 2025 GAITE **Word Segmentation** 과 같은 *시퀀스 라벨링*): 문장의 **각 토큰에 라벨을 다는**
방식. Word Segmentation 은 글자마다 `1=단어 끝`(20번)을 예측했고, 여기선 토큰마다 **BIO 개체명 태그**(O·B_PER·
I_PER·B_LOC·I_LOC·B_ORG·I_ORG)를 예측한다 — 둘 다 **임베딩 + BiLSTM → 토큰별 분류**.

**대회**: [MIPT NER](https://www.kaggle.com/c/mipt-ner) — 러시아어 문장의 **개체명 인식(NER)**. 20번(구성한 과제,
캐글 LB 무관)과 달리 **실제 캐글 대회**이고 누구나 참가 가능해 `id,tag` 형식으로 **진짜 CSV 제출**한다. 데이터도
가볍다(dev 0.43MB · test 0.66MB).

**핵심 흐름 & 배울 점**:
1. **토큰별 BIO 라벨** — `dev.csv` 의 (문장, 태그열)에서 토큰별 학습 데이터 구성.
2. **글자(char) 특징의 중요성** — 데이터가 적고(1.5천 문장) 러시아어 어휘가 희소해 *학습에 없던 단어(OOV)* 가 많다.
   **char-CNN**(접미사·형태) + **대문자 여부** 특징이 OOV 를 살려 NER F1 을 크게 올린다(단어만 vs +char 비교).
3. **단어+글자 임베딩 → BiLSTM → BIO 분류**.
4. 예측 태그를 공백으로 이어 **실제 `submission.csv`** 제출.

> ⚙️ GPU 권장(작은 BiLSTM, T4 ~1분). 러시아어지만 글자/단어 모델이라 언어 무관.
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.
> 🔑 **첫 실행 시 대회 Rules 수락 필요**: 403 이 나면 [대회 페이지](https://www.kaggle.com/c/mipt-ner/rules)
> 에서 **"I Understand and Accept"** 1회 클릭 후 다시 실행하세요(자동화 불가).

## 0. 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle 에서 데이터 다운로드 (dev / test / sample_submission)
403 이 나면 대회 Rules 를 한 번 수락해야 합니다(위 안내 참조).

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("mipt-ner", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:200])
    print("→ 403 이면 https://www.kaggle.com/c/mipt-ner/rules 에서 'I Understand and Accept' 1회 후 재실행")

## 2. 데이터 구성 — 토큰별 BIO 라벨 & 단어/글자 사전
`dev.csv` 는 (태그열, 문장, 플래그) 3열. 문장을 공백으로 토큰화하면 태그 개수와 1:1 로 맞는다.
단어 사전(소문자)과 **글자 사전**(원형 — 대소문자 보존)을 만든다.

In [ ]:
import numpy as np, pandas as pd
TAGS = ["O", "B_PER", "I_PER", "B_LOC", "I_LOC", "B_ORG", "I_ORG"]
t2i = {t: i for i, t in enumerate(TAGS)}

dev = pd.read_csv(os.path.join(WORK_DIR, "dev.csv")); dev.columns = ["tags", "text", "flag"]
data = []
for r in dev.itertuples():
    w, t = str(r.text).split(), str(r.tags).split()
    if len(w) == len(t): data.append((w, t))

WV = {"<pad>": 0, "<unk>": 1}; CV = {"<pad>": 0, "<unk>": 1}; MAXC = 20
for w, _ in data:
    for tok in w:
        WV.setdefault(tok.lower(), len(WV))
        for ch in tok: CV.setdefault(ch, len(CV))

def w_ids(ws):  return [WV.get(t.lower(), 1) for t in ws]
def c_ids(ws):  return [[CV.get(ch, 1) for ch in t[:MAXC]] + [0]*(MAXC - len(t[:MAXC])) for t in ws]
def cap_f(ws):  return [1.0 if t[:1].isupper() else 0.0 for t in ws]
print("문장:", len(data), "| 단어사전:", len(WV), "| 글자사전:", len(CV), "| 태그:", TAGS)

## 3. 배치 구성 (패딩 + 마스크) & train/valid 분할

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = np.random.RandomState(0); idx = rng.permutation(len(data)); cut = int(len(data) * 0.85)
tr_data = [data[i] for i in idx[:cut]]; va_data = [data[i] for i in idx[cut:]]

def batchify(d, bs=32, shuffle=False):
    order = np.random.permutation(len(d)) if shuffle else np.arange(len(d))
    for i in range(0, len(order), bs):
        chunk = [d[j] for j in order[i:i+bs]]; m = max(len(w) for w, _ in chunk)
        X = np.zeros((len(chunk), m), "int64"); C = np.zeros((len(chunk), m, MAXC), "int64")
        CAP = np.zeros((len(chunk), m, 1), "float32"); Y = np.full((len(chunk), m), -100, "int64")
        Msk = np.zeros((len(chunk), m), bool)
        for j, (w, t) in enumerate(chunk):
            X[j, :len(w)] = w_ids(w); C[j, :len(w)] = c_ids(w); CAP[j, :len(w), 0] = cap_f(w)
            Y[j, :len(t)] = [t2i[x] for x in t]; Msk[j, :len(w)] = 1
        yield (torch.from_numpy(X), torch.from_numpy(C), torch.from_numpy(CAP), torch.from_numpy(Y), Msk)
print("train:", len(tr_data), "| valid:", len(va_data))

## 4. 모델 — 단어 임베딩 (+ char-CNN + 대문자) → BiLSTM → BIO
`use_char=False` 면 단어 임베딩만, `True` 면 **char-CNN**(글자→Conv→maxpool로 단어 표현)과 **대문자 플래그**를
더한다. OOV(처음 보는 단어)도 글자/형태로 추론할 수 있어 NER 에 강하다.

In [ ]:
import torch.nn as nn

class NERTagger(nn.Module):
    def __init__(self, n_word, n_char, use_char=True, we=64, ce=30, ch=64, hid=128):
        super().__init__(); self.use_char = use_char
        self.we = nn.Embedding(n_word, we, padding_idx=0)
        in_dim = we
        if use_char:
            self.ce = nn.Embedding(n_char, ce, padding_idx=0)
            self.conv = nn.Conv1d(ce, ch, 3, padding=1)
            in_dim += ch + 1                                   # +char repr, +대문자 플래그
        self.drop = nn.Dropout(0.3)
        self.lstm = nn.LSTM(in_dim, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hid * 2, len(TAGS))
    def forward(self, x, c, cap):
        h = self.we(x)
        if self.use_char:
            B, T, Lc = c.shape
            e = self.ce(c.view(B*T, Lc)).transpose(1, 2)       # (B*T, ce, Lc)
            cf = torch.relu(self.conv(e)).max(-1).values.view(B, T, -1)
            h = torch.cat([h, cf, cap], -1)
        return self.fc(self.drop(self.lstm(h)[0]))

print("params(+char):", sum(p.numel() for p in NERTagger(len(WV), len(CV), True).parameters()))

## 5. 학습 & 평가 — 단어만 vs +char-CNN (토큰 정확도 · 개체 F1)
두 모델을 같은 조건으로 학습해 **char 특징의 효과**를 본다(OOV 때문에 차이가 크다). 평가는 개체 태그(비-O)에
대한 토큰 F1.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
ENT = [t2i[t] for t in TAGS if t != "O"]
EPOCHS = 25

def run(use_char, seed=0):
    torch.manual_seed(seed)
    net = NERTagger(len(WV), len(CV), use_char=use_char).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3); ce = nn.CrossEntropyLoss(ignore_index=-100)
    for epoch in range(EPOCHS):
        net.train()
        for X, C, CAP, Y, _ in batchify(tr_data, shuffle=True):
            opt.zero_grad()
            o = net(X.to(device), C.to(device), CAP.to(device))
            ce(o.reshape(-1, len(TAGS)), Y.to(device).reshape(-1)).backward(); opt.step()
    net.eval(); P, G = [], []
    with torch.no_grad():
        for X, C, CAP, Y, Msk in batchify(va_data, bs=64):
            pr = net(X.to(device), C.to(device), CAP.to(device)).argmax(-1).cpu().numpy()
            for j in range(len(X)):
                n = Msk[j].sum(); P += list(pr[j, :n]); G += list(Y[j, :n].numpy())
    P, G = np.array(P), np.array(G)
    return net, accuracy_score(G, P), f1_score(G, P, labels=ENT, average="micro", zero_division=0)

_,     acc0, f0 = run(use_char=False)
model, acc1, f1 = run(use_char=True)
print(f"단어만        | token acc {acc0:.4f} | 개체 micro-F1 {f0:.4f}")
print(f"+char-CNN+대문자 | token acc {acc1:.4f} | 개체 micro-F1 {f1:.4f}")
print("\n→ 글자 특징이 OOV 를 살려 NER F1 을 크게 올림 — 토큰 시퀀스 라벨링(Word Segmentation 계열) + NER 핵심기법.")

## 6. 캐글 제출 — test 문장 태깅 → `id,tag` (공백 결합)
`test_res.csv` 는 헤더가 없어 `header=None` 으로 읽는다. 각 문장을 태깅해 공백으로 이어 제출한다(샘플과 동일 형식).

In [ ]:
test = pd.read_csv(os.path.join(WORK_DIR, "test_res.csv"), header=None)
model.eval(); rows = []
with torch.no_grad():
    for i in range(len(test)):
        ws = str(test.iloc[i, 0]).split()
        if not ws: rows.append(""); continue
        X = torch.tensor([w_ids(ws)], dtype=torch.long, device=device)
        C = torch.tensor([c_ids(ws)], dtype=torch.long, device=device)
        CAP = torch.tensor([[[v] for v in cap_f(ws)]], dtype=torch.float32, device=device)
        pr = model(X, C, CAP).argmax(-1)[0].cpu().numpy()
        rows.append(" ".join(TAGS[p] for p in pr[:len(ws)]))

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"id": np.arange(len(rows)), "tag": rows}).to_csv(submission_path, index=False)
# 형식 점검: 행수·토큰수 일치
sample = pd.read_csv(os.path.join(WORK_DIR, "crf_sample_submission.csv"))
ok = (len(rows) == len(sample)) and all(len(str(test.iloc[i,0]).split()) == len(rows[i].split()) for i in range(len(rows)))
print("Saved:", submission_path, "| rows:", len(rows), "| sample rows:", len(sample), "| 형식 OK:", ok)
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [MIPT NER](https://www.kaggle.com/c/mipt-ner/submit) 에 제출하세요.

**기법 요약**: Word Segmentation 과 같은 **토큰 단위 시퀀스 라벨링**(임베딩+BiLSTM)을 **실제 캐글 NER 대회**에 적용 —
토큰마다 BIO 태그를 예측해 `id,tag` 제출. 데이터가 적고 OOV 가 많아 **char-CNN + 대문자 특징**이 NER F1 을 크게
끌어올린다(단어만 ≈0.43 → +char ≈0.65). 더 끌어올리려면: **CRF 디코딩**(BIO 전이 제약), 사전학습 임베딩/다국어
Transformer, 글자 BiLSTM, 에폭/하이퍼파라미터 튜닝. (20번=글자 단위 분할 *직접 analog*, 21번=같은 계열 기법의 *실제 캐글 제출*.)